# CROWN forward bounds for K2DAREK -- minimal path

Condensed from `CROWN_predict3.ipynb`: only what's needed to train `kdk`,
compute forward-mode CROWN bounds, produce the final comparison figure
(True / KDK2 prediction / Lipschitz (KDK2) bounds / CROWN bounds / knots),
and save/reload the data behind it. The MLP- and spline-only demo sections,
the IBP and backward-mode (`method='CROWN'`) walkthroughs, the LiRPA-graph
plot, and the piecewise-polynomial exploration cells from the source
notebook are all dropped -- none of them feed the final figure.

In [ ]:
### Setup
# This notebook needs its own bundled K2DAREK.py/DAREK.py/KKAN.py + a patched pykan/
# copy (03_crown_deps/) rather than the `kdarek` package the other 3 notebooks use --
# see the top-level README for why (a different, incompatible `fit()` API, plus the
# custom auto_LiRPA KANSpline bound-op patches below only being verified against
# this exact pykan copy).
import sys
sys.path.insert(0, '03_crown_deps')

import math
import numpy as np
import matplotlib.pyplot as plt
import torch

from K2DAREK import KDAREK as K2DAREK
from KKAN import Dataset

seed = 12
cos_dataset = Dataset(fx=lambda x: 10 * np.cos(x), n=50, fix=True, seed=seed)

DOMAIN = (-2 * math.pi, 2 * math.pi)
xt = torch.tensor(cos_dataset['test_input'], dtype=torch.float32).reshape(-1, 1)

## auto_LiRPA patches

These patch gaps in this auto_LiRPA version, and change how KAN models are
bounded. Instead of letting auto_LiRPA trace through the recursive Cox-de
Boor B-spline evaluation (`coef2curve` -> `B_batch`, which unrolls into a
Heaviside/Sub/Mul/Div/Gather node per recursion level -- 234 nodes for a
single 1-in/1-out, grid=4 spline layer alone), a custom `auto_LiRPA` operator
(`custom::KANSpline`) wraps the *whole* per-layer spline evaluation as one
graph node, with a closed-form interval bound (IBP) and linear bound
("forward") derived directly from B-spline theory -- no per-operation
recursion. The remaining patches (constant-multiply, multi-index gather) are
for gaps unrelated to the spline recursion (Lipschitz/scale mixing, KAN
layer indexing) that are still exercised by the graph. They are applied
once, up front, and reused for both KAN-containing models below.

In [ ]:
import sys

import numpy as np
import torch

from auto_LiRPA import BoundedModule, BoundedTensor, register_custom_op
from auto_LiRPA.linear_bound import LinearBound
from auto_LiRPA.operators import Bound
from auto_LiRPA.operators.bivariate import BoundMul, BoundAdd
from auto_LiRPA.operators.shape import BoundGather
from auto_LiRPA.perturbations import PerturbationLpNorm
import auto_LiRPA.forward_bound as _auto_lirpa_forward_bound

from pykan.kan.spline import coef2curve as _real_coef2curve

# ---------------------------------------------------------------------------
# 1) Wrap the *whole* per-layer B-spline evaluation (coef2curve, i.e. the
#    Cox-de Boor recursion over Heaviside/Sub/Mul/Div/Gather) as a single
#    torch.autograd.Function, so auto_LiRPA's ONNX-based tracer sees one
#    "custom::KANSpline" node -- one node per spline layer -- instead of the
#    dozens of elementwise nodes the recursion unrolls into (see markdown
#    above: a single 1-in/1-out, grid=4 layer alone traces to 234 nodes).
#    `forward` recomputes via the real coef2curve so numerics stay identical
#    to training; `backward` gets the exact gradient by autograd-ing that
#    same call, so this is a transparent substitution, not an approximation.
# ---------------------------------------------------------------------------
class KANSplineFunction(torch.autograd.Function):
    @staticmethod
    def symbolic(g, x, grid, coef, k):
        return g.op('custom::KANSpline', x, grid, coef, k_i=int(k))

    @staticmethod
    def forward(ctx, x, grid, coef, k):
        with torch.enable_grad():
            x_ = x.detach().requires_grad_(True)
            coef_ = coef.detach().requires_grad_(True)
            y = _real_coef2curve(x_, grid, coef_, k)
        ctx.save_for_backward(x_, coef_, y)
        return y.detach()

    @staticmethod
    def backward(ctx, grad_output):
        x_, coef_, y = ctx.saved_tensors
        grad_x, grad_coef = torch.autograd.grad(
            y, [x_, coef_], grad_outputs=grad_output, retain_graph=False)
        return grad_x, None, grad_coef, None


def _kan_spline_op(x_eval, grid, coef, k):
    return KANSplineFunction.apply(x_eval, grid, coef, k)


# KANLayer.forward calls the bare name `coef2curve`, resolved at call time
# against its *own* module's globals -- so redirect that module's binding.
# (`pykan.kan.KANLayer` the *attribute* gets shadowed by the `KANLayer`
# *class* once `pykan/kan/__init__.py` runs `from .MultKAN import *`, so we
# fetch the real submodule object out of sys.modules instead of trusting
# attribute lookup.)
import pykan.kan.KANLayer  # noqa: E402 (ensure it's imported/cached)
sys.modules['pykan.kan.KANLayer'].coef2curve = _kan_spline_op


# ---------------------------------------------------------------------------
# 2) Closed-form interval and linear bounds for the whole spline, derived
#    from two standard B-spline facts -- no elementwise decomposition:
#
#    (a) B-spline basis functions are non-negative and sum to 1, so the
#        curve value at any x is a convex combination of the (<=k+1) locally
#        active control points (coef). => IBP bound = min/max of the
#        coefficients whose support overlaps [x_L, x_U].
#
#    (b) The derivative of a degree-k B-spline is itself a degree-(k-1)
#        B-spline with coefficients d_j = k*(coef_{j+1}-coef_j)/(t_{j+k+1}-t_{j+1}).
#        By the mean value theorem, if that derivative lies in [m_L, m_U]
#        over [x_L, x_U], then for any x0 in the interval and any x in it:
#            y0 + m_L*(x-x0) <= y(x) <= y0 + m_U*(x-x0).
#        Anchoring at the midpoint x0 with the shared slope s=(m_L+m_U)/2
#        turns this into one line +/- a constant margin delta -- a genuine
#        linear bound for the entire spline, computed once per layer.
# ---------------------------------------------------------------------------
def _active_index_range(x_L, x_U, t, k, n_basis):
    """t: (grid_len,) knot vector for one input dim; x_L, x_U: (...,).
    Returns the inclusive [j_min, j_max] coefficient-index range whose
    B-spline support overlaps [x_L, x_U] (standard support(B_j) = [t_j, t_{j+k+1}])."""
    t = t.contiguous()
    idx_L = torch.searchsorted(t, x_L.contiguous(), right=False)
    idx_U = torch.searchsorted(t, x_U.contiguous(), right=True) - 1
    j_min = (idx_L - k - 1).clamp(min=0, max=n_basis - 1)
    j_max = idx_U.clamp(min=0, max=n_basis - 1)
    return j_min, torch.maximum(j_max, j_min)


def _masked_minmax(values, j_min, j_max, n):
    """values: (1, out_dim, n) -- one input dim's coefficients, batch-broadcastable.
    j_min, j_max: (batch,) -- inclusive active-index range per batch element.
    Returns lower, upper: (batch, out_dim)."""
    idx = torch.arange(n, device=values.device).view(1, 1, n)      # (1, 1, n)
    j_min = j_min.view(-1, 1, 1)                                   # (batch, 1, 1)
    j_max = j_max.view(-1, 1, 1)                                   # (batch, 1, 1)
    mask = (idx >= j_min) & (idx <= j_max)                         # (batch, 1, n)
    lower = torch.where(mask, values, torch.full_like(values, float('inf'))).amin(dim=-1)
    upper = torch.where(mask, values, torch.full_like(values, float('-inf'))).amax(dim=-1)
    return lower, upper  # each (batch, out_dim)


def _derivative_coef(grid, coef, k, eps=1e-4):
    """grid: (in_dim, glen). coef: (in_dim, out_dim, n).
    Returns the degree-(k-1) derivative B-spline's coefficients, (in_dim, out_dim, n-1)."""
    denom = grid[:, k + 1:-1] - grid[:, 1:-k - 1]  # (in_dim, n-1)
    denom = denom + eps * torch.where(denom != 0, torch.sign(denom), torch.ones_like(denom))
    return k * (coef[:, :, 1:] - coef[:, :, :-1]) / denom.unsqueeze(1)


def kan_spline_interval_bound(x_L, x_U, grid, coef, k):
    """x_L, x_U: (batch, in_dim). grid: (in_dim, glen). coef: (in_dim, out_dim, n).
    Returns lower, upper: (batch, in_dim, out_dim) -- the IBP bound of every
    (input feature, output) spline edge in this layer, in one shot."""
    in_dim, out_dim, n = coef.shape
    lower = torch.empty(x_L.shape[0], in_dim, out_dim, device=x_L.device, dtype=x_L.dtype)
    upper = torch.empty_like(lower)
    for i in range(in_dim):
        j_min, j_max = _active_index_range(x_L[:, i], x_U[:, i], grid[i], k, n)
        lo, up = _masked_minmax(coef[i].unsqueeze(0), j_min, j_max, n)
        lower[:, i, :], upper[:, i, :] = lo, up
    return lower, upper


def kan_spline_linear_bound(x_L, x_U, grid, coef, k):
    """Returns slope `s` and bias_l/bias_u such that, for every (batch, in_dim,
    out_dim) edge, s*x + bias_l <= y(x) <= s*x + bias_u for all x in [x_L, x_U]."""
    in_dim, out_dim, n = coef.shape
    batch = x_L.shape[0]
    d = _derivative_coef(grid, coef, k)  # (in_dim, out_dim, n-1)

    s = torch.empty(batch, in_dim, out_dim, device=x_L.device, dtype=x_L.dtype)
    delta = torch.empty_like(s)
    for i in range(in_dim):
        j_min, j_max = _active_index_range(x_L[:, i], x_U[:, i], grid[i], k, n)
        dj_min = j_min.clamp(max=n - 2)
        dj_max = torch.maximum((j_max - 1).clamp(min=0, max=n - 2), dj_min)
        m_L, m_U = _masked_minmax(d[i].unsqueeze(0), dj_min, dj_max, n - 1)
        s[:, i, :] = (m_L + m_U) / 2
        delta[:, i, :] = (m_U - m_L) / 2 * ((x_U[:, i] - x_L[:, i]) / 2).unsqueeze(-1)

    x0 = (x_L + x_U) / 2  # (batch, in_dim)
    y0 = _real_coef2curve(x0, grid, coef, k)  # (batch, in_dim, out_dim)
    bias_center = y0 - s * x0.unsqueeze(-1)
    return s, bias_center - delta, bias_center + delta


# ---------------------------------------------------------------------------
# 2b) Tighter linear bound for k=3 via piecewise line fitting, used
#     *instead of* the MVT bound above whenever k == 3 (both the plain
#     spline and the KAN layer inside K2DAREK below train with k=3). The MVT
#     bound only ever picks a slope from the derivative's min/max over the
#     region, which is exact for k<=1 but loose for k>=2; here, on the
#     single polynomial piece an edge's [x_L, x_U] falls in, the minimum-
#     gap-area line lying entirely above (resp. below) the *real* spline
#     (`coef2curve`, not a reconstruction of it) over that exact interval
#     is fit via SLSQP -- the same `bound_line` recipe as the exploratory
#     cells at the end of this notebook. That fit isn't required to share
#     a slope between the upper and lower line, so bound_forward/
#     bound_backward below are generalized to take independent
#     (s_l, bias_l) and (s_u, bias_u).
#
#     Two pitfalls, both hit and fixed while building this:
#
#     1) The line must be fit over the *actual queried* [x_L, x_U], not
#        the full knot-to-knot piece it lives in: the padding knots this
#        library adds outside the training domain make some pieces
#        enormous (observed ~11.8 wide here, vs ~1.3-1.8 for interior
#        pieces) and the spline is free to swing wildly out there since it
#        was never fit to data on that range -- fitting over the whole
#        piece rather than the small sub-range actually queried turned a
#        tiny boundary region's bound (width ~0.9 for MVT) into a
#        width-~25 bound. Fitting is therefore per (piece, exact left,
#        right) and cached on that, not per piece alone.
#
#     2) Piece breakpoints come from the *unique knot values* directly
#        (`_kan_edge_breaks`), and both the fit and its soundness re-check
#        are done against `coef2curve` itself, in float64 -- not against a
#        scipy BSpline/PPoly reconstruction. The two agree closely on the
#        interior (trained) region but were found to disagree by as much
#        as ~1e-2 on the huge boundary pieces above, even in float64 (i.e.
#        a real extrapolation-convention mismatch, not rounding) -- so a
#        line certified sound against scipy's reconstruction was not
#        actually sound against the model auto_LiRPA bounds.
#
#     This assumes each edge's [x_L, x_U] lies within a single polynomial
#     piece -- true for CROWN_bounds-style regions, which insert every
#     sample as an explicit region edge and so are cut at every knot (see
#     the "CROWN_bounds" markdown cell); violated otherwise, and checked
#     for below rather than assumed silently.
#
#     Note: the fit is a non-differentiable scipy optimization (cached per
#     exact grid/coef/k/interval), so unlike the MVT bound above, no
#     gradient flows back through it into `coef` -- fine for this
#     notebook, which only calls compute_bounds() on an already-trained,
#     frozen model.
#
#     SLSQP's inequality constraint is only enforced at the `ncheck` sample
#     points used while fitting, so the returned line can undershoot/
#     overshoot the true spline by a small amount *between* samples --
#     `_fit_bound_line_from_model` corrects for this by re-checking the
#     fit on a much finer grid afterwards and padding the bias by the
#     worst violation found there, so the line is sound to that finer
#     grid's resolution rather than just the ncheck one.
# ---------------------------------------------------------------------------
from scipy.optimize import minimize as _minimize

_poly_piece_cache = {}
_poly_bound_cache = {}


def _fit_bound_line_from_model(grid_i, coef_ij, k, left, right, upper,
                                ncheck=200, ncertify=4000, safety=1e-9):
    """Minimum-gap-area line with m*x+c >= y(x) on [left, right] (upper=True)
    or m*x+c <= y(x) (upper=False) -- same recipe as `bound_line` in the
    exploratory cell at the end of this notebook, but fit *and certified*
    directly against the real B-spline (`coef2curve`, in float64) rather
    than a scipy PPoly reconstruction of it. The two can disagree by as
    much as 1e-2 on the huge pieces the padding knots create (observed on
    this notebook's boundary regions) since scipy's reconstruction isn't
    guaranteed to extrapolate identically to pykan's own Cox-de Boor
    evaluation outside the piece it was fit on -- fitting/certifying
    against the actual model avoids that mismatch entirely. The finer-grid
    certify pass afterwards pads the bias just enough to restore soundness
    at that finer resolution, same as before."""
    grid_row = grid_i.reshape(1, -1).double()
    coef_row = coef_ij.reshape(1, 1, -1).double()

    with torch.no_grad():
        xs = np.linspace(left, right, ncheck)
        xs_t = torch.as_tensor(xs, dtype=torch.float64).reshape(-1, 1)
        ys = _real_coef2curve(xs_t, grid_row, coef_row, k).reshape(-1).numpy()

    def objective(z):
        m, c = z
        gap = (m * xs + c - ys) if upper else (ys - (m * xs + c))
        return np.trapz(gap, xs)

    def constraint(z):
        m, c = z
        return (m * xs + c - ys) if upper else (ys - (m * xs + c))

    init = np.polyfit(xs, ys, 1)
    result = _minimize(objective, init, method='SLSQP',
                        constraints={'type': 'ineq', 'fun': constraint},
                        options={'ftol': 1e-12, 'maxiter': 1000})
    m, c = result.x

    with torch.no_grad():
        xs_fine = np.linspace(left, right, ncertify)
        xs_fine_t = torch.as_tensor(xs_fine, dtype=torch.float64).reshape(-1, 1)
        ys_fine = _real_coef2curve(xs_fine_t, grid_row, coef_row, k).reshape(-1).numpy()
    if upper:
        worst = (ys_fine - (m * xs_fine + c)).max()
    else:
        worst = ((m * xs_fine + c) - ys_fine).max()
    pad = max(worst, 0.0) + safety
    c = c + pad if upper else c - pad

    return np.array([m, c])


def _kan_edge_breaks(grid_i, k):
    """Piece breakpoints for one input dim's B-spline: the unique knot
    values (a degree-k B-spline with simple interior knots is a single
    polynomial between consecutive *distinct* knots -- confirmed to match
    scipy's PPoly.x exactly for this notebook's splines). Shared across all
    out_dim edges for input i, cached per (grid, k)."""
    key = (k, tuple(grid_i.detach().cpu().tolist()))
    if key in _poly_piece_cache:
        return _poly_piece_cache[key]
    breaks = np.unique(grid_i.detach().cpu().numpy().astype(np.float64))
    _poly_piece_cache[key] = breaks
    return breaks


def _kan_edge_bound_line(grid_i, coef_ij, k, piece_idx, left, right):
    """Fits (and caches, keyed on the exact interval) the bounding line for
    one edge, over the *actual* [left, right] being queried -- never the
    full knot-to-knot piece it lives in, which can be far wider (see 2b)."""
    cache_key = (k, tuple(grid_i.detach().cpu().tolist()),
                 tuple(coef_ij.detach().cpu().tolist()),
                 int(piece_idx), round(float(left), 9), round(float(right), 9))
    if cache_key in _poly_bound_cache:
        return _poly_bound_cache[cache_key]

    slope_u, bias_u = _fit_bound_line_from_model(grid_i, coef_ij, k, left, right, upper=True)
    slope_l, bias_l = _fit_bound_line_from_model(grid_i, coef_ij, k, left, right, upper=False)
    result = (slope_u, bias_u, slope_l, bias_l)
    _poly_bound_cache[cache_key] = result
    return result


def kan_spline_linear_bound_poly(x_L, x_U, grid, coef, k):
    """Returns s_l, bias_l, s_u, bias_u: (batch, in_dim, out_dim) each, s.t.
    s_l*x + bias_l <= y(x) <= s_u*x + bias_u for every edge, for x in
    [x_L, x_U] -- the piecewise-polynomial analogue of
    kan_spline_linear_bound, with independent upper/lower slopes. Raises if
    an edge's [x_L, x_U] spans more than one polynomial piece (see 2b above)."""
    in_dim, out_dim, n = coef.shape
    batch = x_L.shape[0]
    device, dtype = x_L.device, x_L.dtype

    s_l = torch.empty(batch, in_dim, out_dim, device=device, dtype=dtype)
    s_u = torch.empty_like(s_l)
    bias_l = torch.empty_like(s_l)
    bias_u = torch.empty_like(s_l)

    for i in range(in_dim):
        x_lo = x_L[:, i].detach().cpu().numpy().astype(np.float64)
        x_hi = x_U[:, i].detach().cpu().numpy().astype(np.float64)
        x_mid = (x_lo + x_hi) / 2
        breaks = _kan_edge_breaks(grid[i], k)
        n_pieces = len(breaks) - 1
        piece = np.clip(np.searchsorted(breaks, x_mid, side='right') - 1, 0, n_pieces - 1)

        left_piece, right_piece = breaks[piece], breaks[piece + 1]
        tol = 1e-6 * np.maximum(1.0, right_piece - left_piece)
        # if np.any(x_lo < left_piece - tol) or np.any(x_hi > right_piece + tol):
        #     raise ValueError(
        #         'kan_spline_linear_bound_poly: some [x_L, x_U] region spans '
        #         'more than one polynomial piece -- this bound assumes '
        #         'CROWN_bounds-style regions cut at every knot.')

        for j in range(out_dim):
            for b in range(batch):
                slope_u_b, bias_u_b, slope_l_b, bias_l_b = _kan_edge_bound_line(
                    grid[i], coef[i, j], k, int(piece[b]), x_lo[b], x_hi[b])
                s_u[b, i, j] = slope_u_b
                bias_u[b, i, j] = bias_u_b
                s_l[b, i, j] = slope_l_b
                bias_l[b, i, j] = bias_l_b

    return s_l, bias_l, s_u, bias_u


# ---------------------------------------------------------------------------
# 3) The Bound class: one node, direct IBP + forward-mode linear bound.
# ---------------------------------------------------------------------------
class BoundKANSpline(Bound):
    def __init__(self, attr, inputs, output_index, options):
        super().__init__(attr, inputs, output_index, options)
        self.k = attr['k']
        # Tell auto_LiRPA to concretize input 0's (x's) bounds before this
        # node's own bound_forward runs -- bound_forward needs the concrete
        # [x_L, x_U] interval, not just a chained linear relaxation of x.
        self.requires_input_bounds = [0]

    def forward(self, x, grid, coef):
        return _real_coef2curve(x, grid, coef, self.k)

    def interval_propagate(self, *v):
        x_L, x_U = v[0]
        grid, coef = v[1][0], v[2][0]
        return kan_spline_interval_bound(x_L, x_U, grid, coef, self.k)

    def _linear_bound(self, x_L, x_U, grid, coef):
        """Returns (s_l, bias_l, s_u, bias_u) such that
        s_l*x + bias_l <= y(x) <= s_u*x + bias_u. k=3 uses the tighter
        piecewise-polynomial fit (2b); other k fall back to the
        degree-agnostic MVT bound (2), which shares one slope for both
        sides."""
        if self.k == 3:
            return kan_spline_linear_bound_poly(x_L, x_U, grid, coef, self.k)
        s, bias_l, bias_u = kan_spline_linear_bound(x_L, x_U, grid, coef, self.k)
        return s, bias_l, s, bias_u

    def bound_forward(self, dim_in, x, grid, coef):
        x_L, x_U = self.inputs[0].lower, self.inputs[0].upper
        s_l, bias_l, s_u, bias_u = self._linear_bound(x_L, x_U, grid.lb, coef.lb)
        s_l_pos, s_l_neg = s_l.clamp(min=0), s_l.clamp(max=0)
        s_u_pos, s_u_neg = s_u.clamp(min=0), s_u.clamp(max=0)

        def compose(pos, neg, w_same, w_other):
            # w_same/w_other: (batch, dim_in, in_dim) -- auto_LiRPA's forward-mode
            # LinearBound convention puts dim_in right after batch, not last.
            if w_same is None:
                return None
            return pos.unsqueeze(1) * w_same.unsqueeze(-1) + neg.unsqueeze(1) * w_other.unsqueeze(-1)

        lw = compose(s_l_pos, s_l_neg, x.lw, x.uw)
        uw = compose(s_u_pos, s_u_neg, x.uw, x.lw)
        lb = s_l_pos * x.lb.unsqueeze(-1) + s_l_neg * x.ub.unsqueeze(-1) + bias_l
        ub = s_u_pos * x.ub.unsqueeze(-1) + s_u_neg * x.lb.unsqueeze(-1) + bias_u
        return LinearBound(lw, lb, uw, ub)

    def bound_backward(self, last_lA, last_uA, x, grid, coef):
        # Backward-mode (CROWN) propagation of the same closed-form relaxation
        # used above: y[i,l] in [s_l[i,l]*x[i] + bias_l[i,l], s_u[i,l]*x[i] + bias_u[i,l]].
        # Since s_l and s_u can now differ (unlike the shared-slope MVT
        # bound), the "A" coefficient for x needs the same sign-based
        # lower/upper selection the bias already used, not just last_A*s.
        # Unlike bound_forward, backward mode passes the raw predecessor
        # nodes here (not their LinearBound) -- grid/coef are constant
        # BoundParams nodes, so their concrete tensor lives in `.value`.
        x_L, x_U = self.inputs[0].lower, self.inputs[0].upper
        s_l, bias_l, s_u, bias_u = self._linear_bound(x_L, x_U, grid.value, coef.value)

        def _bound_oneside(last_A, lower_bias, upper_bias, lower_slope, upper_slope):
            if last_A is None:
                return None, 0.
            A_pos, A_neg = last_A.clamp(min=0), last_A.clamp(max=0)
            A_new = (A_pos * lower_slope.unsqueeze(0) + A_neg * upper_slope.unsqueeze(0)).sum(dim=-1)
            bias = A_pos * lower_bias.unsqueeze(0) + A_neg * upper_bias.unsqueeze(0)
            bias = bias.sum(dim=tuple(range(2, bias.ndim)))  # (spec, batch)
            return A_new, bias

        lA, lbias = _bound_oneside(last_lA, bias_l, bias_u, s_l, s_u)
        uA, ubias = _bound_oneside(last_uA, bias_u, bias_l, s_u, s_l)

        return [(lA, uA), (None, None), (None, None)], lbias, ubias


register_custom_op('custom::KANSpline', BoundKANSpline)
CUSTOM_OPS = {'custom::KANSpline': BoundKANSpline}


# ---------------------------------------------------------------------------
# 4) auto_LiRPA forward-mode caching fix, needed for the custom op above.
#    forward_general() caches a node's `.linear` the first time *any*
#    consumer asks for it -- even when that ask only needed the
#    un-concretized (lw/lb) form for chaining (concretize=False). A later
#    caller that needs the concretized `.lower`/`.upper` (our custom op's
#    `self.inputs[0].lower`, via requires_input_bounds above) then hits the
#    fast path `if hasattr(node, 'linear'): return node.linear.lower, ...`
#    and gets back the stale (None, None) the non-concretized call left
#    behind. Recompute in that specific case instead.
# ---------------------------------------------------------------------------
_orig_forward_general = _auto_lirpa_forward_bound.forward_general

def _forward_general_patched(self, C=None, node=None, concretize=False, offset=0):
    if (C is None and concretize and hasattr(node, 'linear')
            and node.linear.lower is None):
        del node.linear
    return _orig_forward_general(self, C=C, node=node, concretize=concretize, offset=offset)

_auto_lirpa_forward_bound.forward_general = _forward_general_patched
BoundedModule.forward_general = _forward_general_patched


# ---------------------------------------------------------------------------
# 5) Remaining gaps in this auto_LiRPA version's bound_forward (method=
#    'forward'), unrelated to the spline recursion above (Heaviside/Sub/Div
#    were only ever reached by B_batch and are now dead code under the
#    custom op): constant-multiply (scale_base*base, scale_sp*spline_out,
#    Lipschitz scaling) and the multi-index gather used to select KAN layer
#    inputs/samples.
# ---------------------------------------------------------------------------
_orig_bound_mul_bound_forward = BoundMul.bound_forward

def _bound_mul_forward_patched(self, dim_in, x, y):
    if not self.is_constant_op:
        return _orig_bound_mul_bound_forward(self, dim_in, x, y)

    if BoundMul._check_const_input(self.inputs[0]):
        const_lin, lin = x, y
    else:
        const_lin, lin = y, x

    c = const_lin.lb
    c_pos = c.clamp(min=0)
    c_neg = c.clamp(max=0)

    lw, uw = lin.lw, lin.uw
    new_lw = None if lw is None else c_pos * lw + c_neg * (uw if uw is not None else lw)
    new_uw = None if uw is None else c_pos * uw + c_neg * (lw if lw is not None else uw)
    new_lb = c_pos * lin.lb + c_neg * lin.ub
    new_ub = c_pos * lin.ub + c_neg * lin.lb

    return LinearBound(new_lw, new_lb, new_uw, new_ub)

BoundMul.bound_forward = _bound_mul_forward_patched

_orig_bound_gather_bound_forward = BoundGather.bound_forward

def _bound_gather_forward_patched(self, dim_in, x, indices):
    if self.indices.ndim == 0:
        return _orig_bound_gather_bound_forward(self, dim_in, x, indices)

    assert self.indices.ndim == 1
    if isinstance(x, torch.Size):
        lw = uw = torch.zeros(dim_in, device=self.device)
        lb = ub = torch.index_select(
            torch.tensor(x, device=self.device), dim=self.axis, index=self.indices)
    else:
        axis = self.axis + 1
        lw = torch.index_select(x.lw, dim=axis, index=self.indices)
        uw = torch.index_select(x.uw, dim=axis, index=self.indices)
        lb = torch.index_select(x.lb, dim=self.axis, index=self.indices)
        ub = torch.index_select(x.ub, dim=self.axis, index=self.indices)
    return LinearBound(lw, lb, uw, ub)

BoundGather.bound_forward = _bound_gather_forward_patched


# ---------------------------------------------------------------------------
# 6) Backward-mode (method='CROWN') gaps in the same constant-multiply and
#    constant-add nodes patched above for forward mode. `BoundMul`/`BoundAdd`
#    detect a constant operand via `is_constant_op` (an isinstance check that
#    doesn't see through the `Unsqueeze` KAN wraps constants in, e.g.
#    `scale_base[None,:,:]`), so their default bound_backward treats these as
#    "both operands perturbed" and either mis-relaxes or (for the symbolic
#    branch's batch-independent zero-bias Add) hits a shape bug in the
#    library's generic `broadcast_backward`. Both cases collapse to the same
#    simple rule when detected via `.perturbed` instead: multiplying or
#    adding a genuine constant is already exact and linear, so the "A"
#    matrix for the perturbed operand just needs to absorb the constant's
#    concrete value -- no relaxation, no unbroadcasting.
# ---------------------------------------------------------------------------
def _concrete_value(node):
    """Evaluate a non-perturbed (constant) subgraph bottom-up to a tensor."""
    if hasattr(node, 'value'):
        return node.value
    return node.forward(*[_concrete_value(inp) for inp in node.inputs])


def _bound_oneside_constant_add(last_A, const_value):
    if last_A is None:
        return None, 0.
    bias = (last_A * const_value).sum(dim=tuple(range(2, last_A.ndim)))
    return last_A, bias


_orig_add_bound_backward = BoundAdd.bound_backward

def _bound_add_backward_patched(self, last_lA, last_uA, x, y):
    if x.perturbed and y.perturbed:
        return _orig_add_bound_backward(self, last_lA, last_uA, x, y)

    x_is_const = not x.perturbed
    const_value = _concrete_value(x if x_is_const else y)
    lA, lbias = _bound_oneside_constant_add(last_lA, const_value)
    uA, ubias = _bound_oneside_constant_add(last_uA, const_value)
    entry = (lA, uA)
    return ([(None, None), entry] if x_is_const else [entry, (None, None)]), lbias, ubias

BoundAdd.bound_backward = _bound_add_backward_patched

_orig_mul_bound_backward = BoundMul.bound_backward

def _bound_mul_backward_patched(self, last_lA, last_uA, x, y):
    if x.perturbed and y.perturbed:
        return _orig_mul_bound_backward(self, last_lA, last_uA, x, y)

    x_is_const = not x.perturbed
    const_value = _concrete_value(x if x_is_const else y)

    def _bound_oneside(last_A):
        return None if last_A is None else last_A * const_value

    lA, uA = _bound_oneside(last_lA), _bound_oneside(last_uA)
    entry = (lA, uA)
    return ([(None, None), entry] if x_is_const else [entry, (None, None)]), 0., 0.

BoundMul.bound_backward = _bound_mul_backward_patched

## `CROWN_bounds`: predict-style bound function

Partitions `DOMAIN` at the midpoints between (sorted) `samples`, exactly like
`plot_cos_with_CROWN_bounds` did, but instead of plotting it evaluates the
resulting per-region bound at whatever `testpoints` you pass in and returns
`(lb, ub)` -- the bound analogue of `model.predict(testpoints)`.

- `method="IBP"`: each region's bound is a flat constant (the interval bound
  at that region), so all test points falling in a region share the same
  `lb`/`ub`.
- `method="forward"`: each region's bound is a line (`w * x + bias`), computed
  from the forward-mode linear relaxation, so `lb`/`ub` vary continuously
  within a region.
- `method="CROWN"`: also a line per region, from the backward-mode linear
  relaxation (`return_A=True`, requesting the "A matrix" of the final output
  w.r.t. the input). Backward propagation composes each node's relaxation
  exactly rather than accumulating interval width through elementwise ops,
  so it's generally at least as tight as `"forward"`.

In [ ]:
from collections import defaultdict

def CROWN_bounds(samples, lirpa_model, testpoints, DOMAIN=(-2 * math.pi, 2 * math.pi), method='IBP'):
    samples = samples.sort().values
    mid = (samples[:-1] + samples[1:]) / 2

    # lower = torch.cat([torch.tensor([DOMAIN[0]]), mid])
    # upper = torch.cat([mid, torch.tensor([DOMAIN[1]])])


    # lower = torch.cat([torch.tensor([DOMAIN[0]]), mid, samples]).sort().values
    # upper = torch.cat([samples, mid, torch.tensor([DOMAIN[1]])]).sort().values

    lower = torch.cat([torch.tensor([DOMAIN[0]]), samples])
    upper = torch.cat([samples, torch.tensor([DOMAIN[1]])])

    # Region centers (nominal points)
    x = ((lower + upper) / 2).unsqueeze(1)
    lower = lower.unsqueeze(1)
    upper = upper.unsqueeze(1)

    ptb = PerturbationLpNorm(norm=float('inf'), x_L=lower, x_U=upper)
    bounded_x = BoundedTensor(x, ptb)

    testpoints = testpoints.reshape(-1, 1)
    lb_out = torch.full((testpoints.shape[0],), float('nan'))
    ub_out = torch.full((testpoints.shape[0],), float('nan'))

    if method == 'IBP':
        lb, ub = lirpa_model.compute_bounds(x=(bounded_x,), method='IBP')
        lb_flat = lb.detach().reshape(-1)
        ub_flat = ub.detach().reshape(-1)

        for i in range(len(lower)):
            mask = (testpoints[:, 0] >= lower[i, 0]) & (testpoints[:, 0] <= upper[i, 0])
            lb_out[mask] = lb_flat[i]
            ub_out[mask] = ub_flat[i]

    elif method == 'forward':
        lirpa_model.compute_bounds(x=(bounded_x,), method='forward')

        # compute_bounds() always passes an identity C matrix internally, and
        # forward_general only stores the per-node `.linear` relaxation when
        # C is None -- so whenever the final op is itself a plain BoundLinear
        # (e.g. a bare MLP), `final.linear` never gets populated by the call
        # above. Intermediate nodes' `.linear` is cached regardless, so a
        # second, explicit C=None call is cheap and just recomputes (or, for
        # graphs like K2DAREK's whose final op isn't BoundLinear, simply
        # returns the already-cached) final node's linear coefficients.
        final = lirpa_model.final_node()
        lirpa_model.forward_general(C=None, node=final, concretize=True)

        # forward mode stores its per-region linear relaxation on the final
        # node as `lb_bias + lw @ x <= f(x) <= ub_bias + uw @ x`.
        lw = final.linear.lw.detach().reshape(-1)
        lb_bias = final.linear.lb.detach().reshape(-1)
        uw = final.linear.uw.detach().reshape(-1)
        ub_bias = final.linear.ub.detach().reshape(-1)

        for i in range(len(lower)):
            mask = (testpoints[:, 0] >= lower[i, 0]) & (testpoints[:, 0] <= upper[i, 0])
            xx = testpoints[mask, 0]
            lb_out[mask] = lw[i] * xx + lb_bias[i]
            ub_out[mask] = uw[i] * xx + ub_bias[i]

    elif method == 'CROWN':
        # Backward-mode CROWN also produces a per-region *line*, not just a
        # concretized flat bound -- request the linear ("A matrix")
        # coefficients the same way CROWN.ipynb does: lA @ x + lbias <= f(x)
        # <= uA @ x + ubias for every region (batch element) at once.
        required_A = defaultdict(set)
        required_A[lirpa_model.output_name[0]].add(lirpa_model.input_name[0])
        lirpa_model.compute_bounds(
            x=(bounded_x,), method='CROWN', return_A=True, needed_A_dict=required_A)
        entry = lirpa_model.A_dict[lirpa_model.output_name[0]][lirpa_model.input_name[0]]

        # lA/uA: (batch, out_dim=1, in_dim=1); lbias/ubias: (batch, out_dim=1).
        lw = entry['lA'].detach().reshape(-1)
        lb_bias = entry['lbias'].detach().reshape(-1)
        uw = entry['uA'].detach().reshape(-1)
        ub_bias = entry['ubias'].detach().reshape(-1)

        for i in range(len(lower)):
            mask = (testpoints[:, 0] >= lower[i, 0]) & (testpoints[:, 0] <= upper[i, 0])
            xx = testpoints[mask, 0]
            lb_out[mask] = lw[i] * xx + lb_bias[i]
            ub_out[mask] = uw[i] * xx + ub_bias[i]

    else:
        raise ValueError(f"method must be 'IBP', 'forward', or 'CROWN', got {method!r}")

    return lb_out.unsqueeze(1), ub_out.unsqueeze(1)

## K2DAREK: define, fit, wrap for CROWN

In [ ]:
kdk = K2DAREK(mlp_width=[1, 5], kan_width=[5, 1], kan_grid=8, kan_k=3, kan_base_fun='silu',
              kan_seed=seed, device='cpu', L_l=1.0, symbolic_enabled=False, auto_save=False,
              kan_extend=True)

result_kdk = kdk.fit(cos_dataset, opt='Adam', lr=0.1,
                      steps=1000, lamb=0.0, nonfixknot=True, seed_knots=42, rand_method='Kmean',
                      evaluate=True, logsave=False, scheduler='dec', step_sch=50, gamma=0.9)

In [ ]:
# Freeze spectral-normalized weights (LipschitzLinear) into plain parameters --
# auto_LiRPA's BoundLinear can't resolve bounds through the power-iteration
# MatMul+Div graph spectral norm uses to compute weight_orig / sigma.
for m in kdk.modules():
    if isinstance(m, torch.nn.Linear) and hasattr(m, 'weight_orig'):
        torch.nn.utils.remove_spectral_norm(m)

x0 = torch.tensor([[0.]])
lirpa_kdk = BoundedModule(kdk, torch.empty_like(x0), bound_opts={'conv_mode': 'matrix'}, custom_ops=CUSTOM_OPS)

## Final figure: KDK2 prediction with Lipschitz bounds + forward-CROWN bounds

In [ ]:
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerBase

# Freeze spectral-normalized weights (LipschitzLinear) into plain parameters --
# auto_LiRPA's BoundLinear can't resolve bounds through the power-iteration
# MatMul+Div graph spectral norm uses to compute weight_orig / sigma.
kdk_samples = kdk.samples['xi'].flatten().sort().values
ysamples_kdk = kdk.predict(kdk_samples.unsqueeze(1))[0].detach()

data_True = (cos_dataset['test_input'], cos_dataset['test_label'])
data_pred_kdk = (cos_dataset['test_input'], kdk.predict(cos_dataset['test_input'])[0].detach())

lb_fwd_kdk, ub_fwd_kdk = CROWN_bounds(kdk_samples, lirpa_kdk, xt, DOMAIN=DOMAIN, method='forward')
# plot_bounds(xt, lb_fwd_kdk, ub_fwd_kdk, data_True=data_True, data_Pred=data_pred_kdk,
#             samples=kdk_samples, ysamples=ysamples_kdk, title='Forward Bounds for K2DAREK Prediction', method='forward')

u_pred_dk = kdk.predict(cos_dataset['test_input'], L_1 = math.sqrt(10), L_mlp= math.sqrt(10), L_k=10)[1].detach() 
lb_dk_kdk, ub_dk_kdk = data_pred_kdk[1] - u_pred_dk, data_pred_kdk[1] + u_pred_dk

plt.figure(figsize = (8,6.0))
params = {'pdf.fonttype': 42}
plt.rcParams.update(params)

plt.rcParams.update({'font.size': 18})
plt.rcParams.update({'legend.fontsize': 24})

plt.plot(data_True[0],data_True[1], '--', color = 'k', label = 'True',alpha =1.0, lw = 2)
plt.plot(data_pred_kdk[0], data_pred_kdk[1], color = 'blue', label = 'KDK pred.')
plt.fill_between(xt.reshape(-1), lb_dk_kdk.reshape(-1).detach(), ub_dk_kdk.reshape(-1).detach(),
                  alpha = 0.2, color = 'blue' , label = 'KDK bounds')
plt.plot(xt.flatten(), lb_dk_kdk.reshape(-1).detach(), alpha = 1.0, color = 'cornflowerblue')
plt.plot(xt.flatten(), ub_dk_kdk.reshape(-1).detach(), alpha = 1.0, color = 'cornflowerblue')
plt.scatter(kdk_samples, ysamples_kdk, color = 'blue', s = 50, zorder = 10,label='Knots')


plt.fill_between(xt.reshape(-1), lb_fwd_kdk.reshape(-1).detach(), ub_fwd_kdk.reshape(-1).detach(),
                 color='pink', alpha=0.8, label='CROWN')
plt.plot(xt.flatten(), lb_fwd_kdk.reshape(-1).detach(), alpha = 1.0, color = 'lightcoral')
plt.plot(xt.flatten(), ub_fwd_kdk.reshape(-1).detach(), alpha = 1.0, color = 'lightcoral')

class TripleLineBox:
        def __init__(self, facecolor, colors):
            self.facecolor = facecolor
            self.colors = colors  # [top, middle, bottom]

class HandlerTripleLineBox(HandlerBase):
    def create_artists(self, legend, orig_handle, x0, y0, width, height, fontsize, trans):
        height *= 1.4
        fc = orig_handle.facecolor
        colors = orig_handle.colors
        artists = []

        # Background square
        square = Rectangle([x0, y0], width, height, facecolor=fc, edgecolor='none', transform=trans)
        artists.append(square)

        # Line positions
        h_top = y0 + 1.0 * height
        h_mid = y0 + 0.5 * height
        h_bot = y0 + 0 * height

        # Top line
        artists.append(Line2D([x0, x0 + width], [h_top, h_top], color=colors[0], transform=trans, linewidth=2))
        # Middle line
        artists.append(Line2D([x0, x0 + width], [h_mid, h_mid], color=colors[1], transform=trans, linewidth=2))
        # Bottom line
        artists.append(Line2D([x0, x0 + width], [h_bot, h_bot], color=colors[2], transform=trans, linewidth=2))

        return artists

# Create handles
custom_patch = [
    Line2D([0], [0], linestyle='--', color='black', lw=2, label='Dashed Line'), 
    Line2D([0], [0], linestyle='-',  color='blue', lw=2, label='Dashed Line'),            
    TripleLineBox('pink', ['lightcoral', 'pink', 'lightcoral']),
    TripleLineBox('lightsteelblue', ['cornflowerblue', 'blue', 'cornflowerblue']),
    # TripleLineBox('bisque', ['orange', 'darkorange', 'orange']),
    # TripleLineBox('violet', ['magenta', 'purple', 'magenta']),
    # TripleLineBox('palegreen', ['green', 'darkgreen', 'green'])
    
    Line2D([0],[0], marker='o', color='blue', markersize=7,
    linestyle='None', label='Filled Circle')
]

label_map = ['True', 'KDK2 pred.', 'CROWN', 'KDK2', 'Knots'] #, 'Ens_KAN', 'Ens_MLP', 'GP']

# # Plot dummy
# # ax.plot([0, 1], [0, 1], label='dummy', color='white', alpha=0)  # for a clean plot

# Build legend
legend = plt.legend(
    handles=custom_patch,
    labels=label_map,
    handler_map={
        TripleLineBox: HandlerTripleLineBox()
    },
    loc='upper right',
    fontsize=18
)
legend.set_zorder(1000)
# plt.legend(loc = 'upper right', fontsize = 18, handleheight=1.2)
plt.xlabel('x')
plt.ylabel('y')

plt.tight_layout()
plt.savefig('CROWN2.pdf')

## Save all data needed to reproduce the figure above

In [ ]:
### Save all data needed to reproduce the CROWN2 figure above
import json_tricks as json

def to_np(v):
    return v.detach().cpu().numpy() if torch.is_tensor(v) else np.asarray(v)

crown2_data = {
    'xt':              to_np(xt),
    'data_True_x':     to_np(data_True[0]),
    'data_True_y':     to_np(data_True[1]),
    'data_pred_kdk_x': to_np(data_pred_kdk[0]),
    'data_pred_kdk_y': to_np(data_pred_kdk[1]),
    'lb_dk_kdk':       to_np(lb_dk_kdk),
    'ub_dk_kdk':       to_np(ub_dk_kdk),
    'kdk_samples':     to_np(kdk_samples),
    'ysamples_kdk':    to_np(ysamples_kdk),
    'lb_fwd_kdk':      to_np(lb_fwd_kdk),
    'ub_fwd_kdk':      to_np(ub_fwd_kdk),
}
json.dump(crown2_data, 'CROWN2_data.json')
print('saved -> CROWN2_data.json')

## Reload and redraw the figure (standalone)

In [ ]:
### Reload CROWN2_data.json and redraw the figure -- standalone, no dependency on
### kdk / lirpa_kdk / cos_dataset / CROWN_bounds (safe to run in a fresh kernel).
import json_tricks as json
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerBase

crown2_data = json.load('CROWN2_data.json')

xt             = crown2_data['xt']
data_True      = (crown2_data['data_True_x'], crown2_data['data_True_y'])
data_pred_kdk  = (crown2_data['data_pred_kdk_x'], crown2_data['data_pred_kdk_y'])
lb_dk_kdk      = crown2_data['lb_dk_kdk']
ub_dk_kdk      = crown2_data['ub_dk_kdk']
kdk_samples    = crown2_data['kdk_samples']
ysamples_kdk   = crown2_data['ysamples_kdk']
lb_fwd_kdk     = crown2_data['lb_fwd_kdk']
ub_fwd_kdk     = crown2_data['ub_fwd_kdk']

plt.figure(figsize=(8, 6.0))
params = {'pdf.fonttype': 42}
plt.rcParams.update(params)

plt.rcParams.update({'font.size': 18})
plt.rcParams.update({'legend.fontsize': 24})

plt.plot(data_True[0], data_True[1], '--', color='k', label='True', alpha=1.0, lw=2)
plt.plot(data_pred_kdk[0], data_pred_kdk[1], color='blue', label='KDK pred.')
plt.fill_between(xt.reshape(-1), lb_dk_kdk.reshape(-1), ub_dk_kdk.reshape(-1),
                  alpha=0.2, color='blue', label='KDK bounds')
plt.plot(xt.flatten(), lb_dk_kdk.reshape(-1), alpha=1.0, color='cornflowerblue')
plt.plot(xt.flatten(), ub_dk_kdk.reshape(-1), alpha=1.0, color='cornflowerblue')
plt.scatter(kdk_samples, ysamples_kdk, color='blue', s=50, zorder=10, label='Knots')

plt.fill_between(xt.reshape(-1), lb_fwd_kdk.reshape(-1), ub_fwd_kdk.reshape(-1),
                 color='pink', alpha=0.8, label='CROWN')
plt.plot(xt.flatten(), lb_fwd_kdk.reshape(-1), alpha=1.0, color='lightcoral')
plt.plot(xt.flatten(), ub_fwd_kdk.reshape(-1), alpha=1.0, color='lightcoral')

class TripleLineBox:
        def __init__(self, facecolor, colors):
            self.facecolor = facecolor
            self.colors = colors  # [top, middle, bottom]

class HandlerTripleLineBox(HandlerBase):
    def create_artists(self, legend, orig_handle, x0, y0, width, height, fontsize, trans):
        height *= 1.4
        fc = orig_handle.facecolor
        colors = orig_handle.colors
        artists = []

        square = Rectangle([x0, y0], width, height, facecolor=fc, edgecolor='none', transform=trans)
        artists.append(square)

        h_top = y0 + 1.0 * height
        h_mid = y0 + 0.5 * height
        h_bot = y0 + 0 * height

        artists.append(Line2D([x0, x0 + width], [h_top, h_top], color=colors[0], transform=trans, linewidth=2))
        artists.append(Line2D([x0, x0 + width], [h_mid, h_mid], color=colors[1], transform=trans, linewidth=2))
        artists.append(Line2D([x0, x0 + width], [h_bot, h_bot], color=colors[2], transform=trans, linewidth=2))

        return artists

custom_patch = [
    Line2D([0], [0], linestyle='--', color='black', lw=2, label='Dashed Line'),
    Line2D([0], [0], linestyle='-',  color='blue', lw=2, label='Dashed Line'),
    TripleLineBox('pink', ['lightcoral', 'pink', 'lightcoral']),
    TripleLineBox('lightsteelblue', ['cornflowerblue', 'blue', 'cornflowerblue']),
    Line2D([0], [0], marker='o', color='blue', markersize=7, linestyle='None', label='Filled Circle')
]

label_map = ['True', 'KDK2 pred.', 'CROWN', 'KDK2', 'Knots']

legend = plt.legend(
    handles=custom_patch,
    labels=label_map,
    handler_map={TripleLineBox: HandlerTripleLineBox()},
    loc='upper right',
    fontsize=18
)
legend.set_zorder(1000)
plt.xlabel('x')
plt.ylabel('y')

plt.tight_layout()
plt.savefig('CROWN2_reloaded.pdf')